# RideFlow — Pipeline Driver (parameterized)

One notebook that runs **any domain / system / layer**, chosen by parameters a Task
passes via `EXECUTE NOTEBOOK NB_PIPELINE('<domain>','<system>','<layer>')` and read from
`sys.argv`. Runs the same `runtime/` code as the local deploy. This replaces the five
per-layer notebooks — the Snowflake analogue of a parameterized Databricks/Fabric notebook.


In [ ]:
# Headless task runs don't have the internal PyPI mirror, so install LakeLogic + pure-
# python deps from wheels staged in Snowflake (@PIPELINE.WHEELS) with pip --no-index.
import os, subprocess, sys
from snowflake.snowpark.context import get_active_session
_session = get_active_session()
os.makedirs('/tmp/wl', exist_ok=True)
_session.file.get('@RIDEFLOW_DEV_DEMO.PIPELINE.WHEELS', '/tmp/wl')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
                '--find-links', '/tmp/wl', '--no-deps', 'lakelogic', 'loguru', 'sqlglot'], check=True)


In [ ]:
import os, sys
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, 'domains_rideflow')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)

from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection

# ---- Parameters -----------------------------------------------------------------
# A Task passes these via  EXECUTE NOTEBOOK NB_PIPELINE('marketplace','rideflow','bronze')
# and the notebook reads them from sys.argv (strings only). Interactive runs (no args)
# fall back to the defaults below — the Snowflake equivalent of notebook widgets.
_args  = sys.argv[1:]
DOMAIN = _args[0] if len(_args) > 0 else 'marketplace'
SYSTEM = _args[1] if len(_args) > 1 else 'rideflow'
LAYER  = _args[2] if len(_args) > 2 else 'all'   # provision|landing|bronze|silver|gold|all
DB     = _args[3] if len(_args) > 3 else 'RIDEFLOW_DEV_DEMO'
SYS    = f'domains_rideflow/{DOMAIN}/{SYSTEM}/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB

# Current DB + schema so the engine's TEMP tables have a home (see notebooks in Workspaces).
session.sql(f'USE DATABASE {DB}').collect()
session.sql(f'USE SCHEMA {DB}.PIPELINE').collect()
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)
print(f'domain={DOMAIN} system={SYSTEM} layer={LAYER} db={DB}')


In [ ]:
from runtime.medallion import run_layer
from runtime.landing import load_system_landing
import pandas as pd

def run_one(layer):
    if layer == 'provision':
        for s in ['LANDING', 'QUARANTINE', DOMAIN.upper(), 'PIPELINE']:
            session.sql(f'CREATE SCHEMA IF NOT EXISTS {DB}.{s}').collect()
        return [('schemas', 'ok', 0, 0)]
    if layer == 'landing':
        return [(t, 'ok', n, 0) for t, n in load_system_landing(conn, DB, DOMAIN, SYSTEM, via_stage=True)]
    return run_layer(conn, DB, SYS, layer)

layers = ['provision', 'landing', 'bronze', 'silver', 'gold'] if LAYER == 'all' else [LAYER]
rows = [(L,) + tuple(r) for L in layers for r in run_one(L)]
pd.DataFrame(rows, columns=['layer', 'entity', 'status', 'good', 'quarantined'])
